In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.pitch import inShotCone

In [2]:
shots = pd.read_parquet("../data/clean/shots.parquet", columns=["id", "x", "y", "shot_freeze_frame"])
features = pd.read_parquet("../data/features/shots_features.parquet")

print("shots:", len(shots))
print("feature rows:", len(features))

shots: 99746
feature rows: 99746


In [3]:
# left merge keeps the features file's row order, so row i here is row i there

frames = features[["id"]].merge(shots, on="id", how="left", validate="one_to_one")

print("rows:", len(frames))
print("shots with no frame:", frames["shot_freeze_frame"].isna().sum())

rows: 99746
shots with no frame: 0


In [4]:
sizes = []
for frame in frames["shot_freeze_frame"]:
    sizes.append(len(frame))
sizes = np.array(sizes)

print("fewest players:", sizes.min())
print("most players:", sizes.max())
print("median:", np.median(sizes))

fewest players: 1
most players: 21
median: 13.0


In [5]:
maxPlayers = 22

# a frame has 21 players at most, so 22 rows fits every shot
def playerTable(frame, shotX, shotY):
    """Table of the players in one shot's frame, and a mask of which rows are real."""
    # each row: x, y, distance to shooter, in cone, teammate, keeper
    table = np.zeros((maxPlayers, 6))
    mask = np.zeros(maxPlayers)

    for i, player in enumerate(frame):
        x = player["location"][0]
        y = player["location"][1]
        distance = np.sqrt((x - shotX) ** 2 + (y - shotY) ** 2)
        cone = int(inShotCone(shotX, shotY, x, y))
        teammate = int(player["teammate"])
        keeper = int(player["position"]["name"] == "Goalkeeper")

        # divided by the pitch size so everything is roughly 0 to 1, the network trains better that way
        table[i] = [x / 120, y / 80, distance / 120, cone, teammate, keeper]
        mask[i] = 1

    return table, mask

In [6]:
table, mask = playerTable(frames["shot_freeze_frame"][0], frames["x"][0], frames["y"][0])
print(int(mask.sum()), "players")
print(table[:4].round(2))

10 players
[[0.74 0.68 0.15 0.   1.   0.  ]
 [0.74 0.72 0.15 0.   0.   0.  ]
 [0.76 0.62 0.14 0.   0.   0.  ]
 [0.88 0.67 0.02 0.   0.   0.  ]]


In [7]:
players = np.zeros((len(frames), maxPlayers, 6))
masks = np.zeros((len(frames), maxPlayers))

for i in range(len(frames)):
    table, mask = playerTable(frames["shot_freeze_frame"][i], frames["x"][i], frames["y"][i])
    players[i] = table
    masks[i] = mask

print(players.shape, masks.shape)

(99746, 22, 6) (99746, 22)


In [8]:
for i in range(len(frames)):
    assert masks[i].sum() == sizes[i]

    count = 0
    for row in players[i]:
        # row is x, y, distance, cone, teammate, keeper
        if row[3] == 1 and row[4] == 0 and row[5] == 0:
            count += 1
    assert count == features["defendersInCone"][i]

In [9]:
ids = frames["id"].to_numpy(dtype=str)
np.savez_compressed("../data/features/player_tables.npz", players=players, masks=masks, ids=ids)
print("saved")

saved
